**AICE Associate <font color=red>모의고사 1회 정답지</font>**

> 문제 노트북과 함께 학습용으로 사용하세요. 실제 시험에서는 정답 셀을 보지 말고 먼저 풀어보세요.


In [ ]:
# 공통 준비
import platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

os_name = platform.system()
if os_name == 'Linux':
    plt.rc('font', family='NanumGothic')
elif os_name == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False


### 1번


In [ ]:
import pandas as pd

### 2번


In [ ]:
df_demographics = pd.read_csv('student_demographics.csv')
df_grades = pd.read_json('course_grades.json')
display(df_demographics.head())
display(df_grades.head())


### 3번


In [ ]:
df = pd.merge(df_demographics, df_grades, on='StudentID', how='inner')
print(df.head())
print(df.info())


### 4번


In [ ]:
sns.countplot(x='ParentalEducation', data=df, order=df['ParentalEducation'].value_counts().index)
plt.title('Distribution of ParentalEducation')
plt.show()

df = df[df['ParentalEducation'] != 'N/A'].copy()


### 5번


In [ ]:
sns.jointplot(x='StudyHours_per_week', y='FinalScore', data=df)
plt.show()

답안5 = df[df['StudyHours_per_week'] >= 50].shape[0]
print(답안5)


### 6번


In [ ]:
df_cleaned = df[df['StudyHours_per_week'] < 50].copy()
df_cleaned = df_cleaned.drop('StudentID', axis=1)
print(df_cleaned.info())


### 7-1번


In [ ]:
print(df_cleaned.isnull().sum())
답안7_1 = int(df_cleaned.isnull().sum().sum())
print(답안7_1)


### 7-2번


In [ ]:
df_processed = df_cleaned.dropna().copy()
df_processed['Absences'] = df_processed['Absences'].astype(int)
print(df_processed.isnull().sum())
print(df_processed.info())


### 8번


In [ ]:
# blank8-1 = get_dummies / blank8-2 = columns
df_encoded = df_processed.drop('Age', axis=1).copy()
df_encoded = pd.get_dummies(df_encoded, columns=['Gender', 'ParentalEducation'])
df_encoded.head()


### 9번


In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('FinalScore', axis=1)
y = df_encoded['FinalScore']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)


### 10번


In [ ]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
print(X_train_scaled.shape, X_valid_scaled.shape)


### 11-1번


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
gbr_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbr_model.fit(X_train_scaled, y_train)


### 11-2번


In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
xgb_model.fit(X_train_scaled, y_train)


### 12번


In [ ]:
from sklearn.metrics import mean_absolute_error

gbr_pred = gbr_model.predict(X_valid_scaled)
xgb_pred = xgb_model.predict(X_valid_scaled)

gbr_mae = mean_absolute_error(y_valid, gbr_pred)
xgb_mae = mean_absolute_error(y_valid, xgb_pred)

print(gbr_mae)
print(xgb_mae)

답안12 = 'GradientBoostingRegressor' if gbr_mae < xgb_mae else 'XGBoostRegressor'
print(답안12)


### 13번


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

tf.random.set_seed(42)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae', 'mse'])
history = model.fit(
    X_train_scaled, y_train,
    epochs=30, batch_size=16,
    validation_data=(X_valid_scaled, y_valid),
    verbose=0
)


### 14번


In [ ]:
# blank14-1 = history.history / blank14-2 = legend
plt.plot(history.history['mse'], label='Training MSE')
plt.plot(history.history['val_mse'], label='Validation MSE')
plt.title('Model MSE over Epochs')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.legend()
plt.show()


### 15번


In [ ]:
dl_pred = model.predict(X_valid_scaled)
dl_mae = mean_absolute_error(y_valid, dl_pred)

답안15 = 'XGBoostRegressor' if xgb_mae < dl_mae else 'DeepLearning'
print('xgb_mae:', xgb_mae, 'dl_mae:', dl_mae)
print(답안15)
